# NB14 — Cross-Validation and Model Selection

> **Never use train R² alone to select a model. Use CV.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)
n = 150
X = np.sort(np.random.uniform(-3, 3, n)).reshape(-1, 1)
y = np.sin(X.ravel()) + np.random.normal(0, 0.4, n)

# Compare models using 5-fold CV
models = {
    'Linear (deg=1)':   Pipeline([('poly', PolynomialFeatures(1)), ('ols', LinearRegression())]),
    'Poly deg=3':       Pipeline([('poly', PolynomialFeatures(3)), ('ols', LinearRegression())]),
    'Poly deg=10':      Pipeline([('poly', PolynomialFeatures(10)), ('ols', LinearRegression())]),
    'Poly deg=10+Ridge':Pipeline([('poly', PolynomialFeatures(10)),
                                   ('sc', StandardScaler()),
                                   ('ridge', Ridge(alpha=1.0))]),
}

print(f"{'Model':<25}  {'CV R² mean':>10}  {'CV R² std':>10}  {'Train R²':>10}")
print("-"*65)
for name, m in models.items():
    cv = cross_val_score(m, X, y, cv=5, scoring='r2')
    m.fit(X, y)
    tr_r2 = m.score(X, y)
    print(f"{name:<25}  {cv.mean():>10.4f}  {cv.std():>10.4f}  {tr_r2:>10.4f}")


In [ ]:
# Learning curves — diagnose bias vs variance
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
selected = {
    'Linear (underfitting)': Pipeline([('poly', PolynomialFeatures(1)), ('ols', LinearRegression())]),
    'Poly deg=3 (good fit)': Pipeline([('poly', PolynomialFeatures(3)), ('ols', LinearRegression())]),
    'Poly deg=10 (overfit)': Pipeline([('poly', PolynomialFeatures(10)), ('ols', LinearRegression())]),
}

for ax, (name, m) in zip(axes, selected.items()):
    sizes, tr_sc, te_sc = learning_curve(
        m, X, y, cv=5, scoring='r2',
        train_sizes=np.linspace(0.1, 1.0, 12))

    tr_mean = tr_sc.mean(axis=1); tr_std = tr_sc.std(axis=1)
    te_mean = te_sc.mean(axis=1); te_std = te_sc.std(axis=1)

    ax.fill_between(sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color='steelblue')
    ax.fill_between(sizes, te_mean-te_std, te_mean+te_std, alpha=0.15, color='crimson')
    ax.plot(sizes, tr_mean, 'o-', color='steelblue', label='Train')
    ax.plot(sizes, te_mean, 's-', color='crimson',   label='CV')
    ax.set_title(name); ax.set_xlabel('Training set size')
    ax.set_ylabel('R²'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Learning curves: diagnose bias/variance trade-off', fontsize=12)
plt.tight_layout(); plt.show()


## Reading learning curves

| Pattern | Diagnosis | Fix |
|---------|-----------|-----|
| Both curves low, close together | High bias (underfitting) | More features, higher degree |
| Train high, CV much lower | High variance (overfitting) | More data, regularisation |
| Both curves converge high | Good model | Ship it |


In [ ]:
# Nested CV: unbiased estimate of generalisation performance
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge
import numpy as np

# Inner CV: tune hyperparameter
# Outer CV: estimate true test performance
param_grid = {'alpha': np.logspace(-3, 3, 30)}
inner_cv = KFold(n_splits=5, shuffle=True, random_state=1)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=2)

ridge_gs = GridSearchCV(Ridge(), param_grid, cv=inner_cv, scoring='r2')
nested_scores = cross_val_score(ridge_gs, X, y, cv=outer_cv, scoring='r2')

print(f"Nested CV R²: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}")
print("This is an unbiased estimate of how well the model will do on new data.")
print("(Simple CV can overestimate if you tune hyperparameters using the same fold.)")


## Key Takeaways

| Technique | Use for |
|-----------|---------|
| k-fold CV | Selecting models / hyperparameters |
| Learning curve | Diagnosing bias vs variance |
| Nested CV | Unbiased performance estimate when tuning hyperparameters |
| Never | Use train R² alone to report model quality |

**Next:** NB15 — Gradient Descent: implement OLS without the normal equations.
